# Run a workflow

In [ ]:
import os
import shutil
import logging

from ewokscore.variable import value_from_transfer
from ewokscore.tests.utils import show_graph
from ewokscore.tests.examples.graphs import get_graph

## Bindings for task schedulers

In [ ]:
schedulers = {}

In [ ]:
def sequential_execution(taskgraph, **kw):
    # runs in a single thread
    from ewokscore import execute_graph

    return execute_graph(taskgraph, **kw)


schedulers[None] = sequential_execution

In [ ]:
def pypushflow_scheduler(taskgraph, **kw):
    # tasks are distributed over processes
    from ewoksppf import execute_graph

    logging.getLogger("pypushflow").setLevel(logging.WARNING)
    logging.getLogger("ewoksppf").setLevel(logging.WARNING)
    return execute_graph(taskgraph, **kw)


schedulers["ppf"] = pypushflow_scheduler

In [ ]:
def dask_scheduler(taskgraph, **kw):
    # tasks are distributed by local or centralized scheduler
    from ewoksdask import execute_graph

    return execute_graph(taskgraph, **kw)


schedulers["dask"] = dask_scheduler

### Select and configure a task scheduler

In [ ]:
# Select scheduler
varinfo = {"root_uri": None}
scheduler = schedulers[None]

# Schedulers options
scheduler_options = {"varinfo": varinfo}
if scheduler is dask_scheduler:
    local_scheduler = True
    if local_scheduler:
        scheduler_options["scheduler"] = "multithreading"
    else:
        # Run this on any host:
        #  >>> from ewoksdask.schedulers import local_scheduler
        #  >>> scheduler = local_scheduler(n_workers=5)
        #
        # Or run this on slurm-access or any slurm node:
        #  >>> from ewoksdask.schedulers import slurm_scheduler
        #  >>> scheduler = slurm_scheduler(maximum_jobs=5)
        scheduler_options["scheduler"] = {"address": "160.103.229.166:38745"}

## Execute a workflow

### Select a task graph

In [ ]:
taskgraph, expected_results = get_graph("acyclic1")
show_graph(taskgraph)

### Prepare directory for persistent results (if any)

In [ ]:
def prepare_results(clean=True):
    if not varinfo["root_uri"]:
        return  # results are not persisted by ewoks
    if clean:
        shutil.rmtree(varinfo["root_uri"], ignore_errors=True)
    os.makedirs(varinfo["root_uri"], exist_ok=True)


prepare_results(clean=True)

### Execute the workflow

In [ ]:
print("Scheduler options:")
print(scheduler_options)
results = scheduler(taskgraph, **scheduler_options)

print("\nResults:")
if scheduler is schedulers[None]:
    for task_name, task in sorted(results.items()):
        for name, result in task.output_values.items():
            print(f" {task_name} {name}:", result)
elif scheduler is schedulers["ppf"]:
    for name, result in results.items():
        if name == "_noinput":
            continue
        value = value_from_transfer(result, varinfo=varinfo)
        print(f" {name}:", value)
else:
    for task_name, task in sorted(results.items()):
        for name, result in task.items():
            value = value_from_transfer(result, varinfo=varinfo)
            print(f" {task_name} {name}:", value)